In [1]:
import pandas as pd
import pandasql as ps
import pixiedust

Pixiedust database opened successfully


In [2]:
import util

In [3]:
util.showdatabases(spark)

+--------------------+
|        databaseName|
+--------------------+
|    covid_2020_q2jun|
|       covid_2020_q3|
|       covid_2020_q4|
|       covid_2021_q1|
|       covid_2021_q2|
|       covid_2021_q3|
|       covid_2021_q4|
|       covid_2022_q1|
|       covid_2022_q2|
|             default|
|           elligo_db|
|        omop_2021_q4|
|       omop_dec_2021|
|real_world_data_d...|
|real_world_data_j...|
|real_world_data_j...|
|real_world_data_m...|
|real_world_data_s...|
+--------------------+



In [4]:
util.usedatabase(spark, "real_world_data_jun_2022")

Using real_world_data_jun_2022 ....


In [5]:
util.showtables(spark)

+--------------------+--------------------+-----------+
|            database|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|real_world_data_j...|             allergy|      false|
|real_world_data_j...|      clinical_event|      false|
|real_world_data_j...|           condition|      false|
|real_world_data_j...|  dedupedemographics|      false|
|real_world_data_j...|        demographics|      false|
|real_world_data_j...|           encounter|      false|
|real_world_data_j...|        immunization|      false|
|real_world_data_j...|                 lab|      false|
|real_world_data_j...|         measurement|      false|
|real_world_data_j...|          medication|      false|
|real_world_data_j...|medication_admini...|      false|
|real_world_data_j...|          order_list|      false|
|real_world_data_j...|        problem_list|      false|
|real_world_data_j...|           procedure|      false|
|real_world_data_j...|provider_demograp...|     

In [6]:
tables = spark.catalog.listTables('real_world_data_jun_2022')
table_names = [table.name for table in tables]
print(table_names)

['allergy', 'clinical_event', 'condition', 'dedupedemographics', 'demographics', 'encounter', 'immunization', 'lab', 'measurement', 'medication', 'medication_administration', 'order_list', 'problem_list', 'procedure', 'provider_demographics', 'questionnaire', 'tableid', 'tenant_attributes']


In [7]:
table_schema = spark.sql(f"DESCRIBE {'real_world_data_jun_2022'}.{'condition'}")

# Show the schema
table_schema.show(truncate=False)

+-----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name               |data_type                                                                                                                                                                 |comment|
+-----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|conditionid            |string                                                                                                                                                                    |null   |
|personid               |string                                                                                                                                                     

In [8]:
table_schema = spark.sql(f"DESCRIBE {'real_world_data_jun_2022'}.{'medication'}")

# Show the schema
table_schema.show(truncate=False)

+-----------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name               |data_type                                                                                                                                                          |comment|
+-----------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|medicationid           |string                                                                                                                                                             |null   |
|encounterid            |string                                                                                                                                                             |null   |
|personid 

In [9]:
table_schema = spark.sql(f"DESCRIBE {'real_world_data_jun_2022'}.{'dedupedemographics'}")

# Show the schema
table_schema.show(truncate=False)

+-------------+----------------------------------------+-------+
|col_name     |data_type                               |comment|
+-------------+----------------------------------------+-------+
|personid     |string                                  |null   |
|tenant       |int                                     |null   |
|birthdate    |struct<array:array<string>,value:string>|null   |
|zipcode      |struct<array:array<string>,value:string>|null   |
|gender       |struct<array:array<string>,value:string>|null   |
|birthsex     |struct<array:array<string>,value:string>|null   |
|maritalstatus|struct<array:array<string>,value:string>|null   |
|race         |struct<array:array<string>,value:string>|null   |
|ethnicity    |struct<array:array<string>,value:string>|null   |
+-------------+----------------------------------------+-------+



In [10]:
a=spark.sql("""
select gender, zipcode from dedupedemographics where personid = '1f52d26b-acd3-4f55-a95b-9a5cc9a52030'""")
a.show(10, truncate=False)

+--------------+--------+
|gender        |zipcode |
+--------------+--------+
|[[Male], Male]|[[9], 9]|
+--------------+--------+



In [15]:
a = spark.sql("""
SELECT *
FROM dedupedemographics
ORDER BY personid
LIMIT 5
""")
a.show(truncate=False)


+------------------------------------+------+--------------------------+--------+------------------+------------------+--------------------------------+-------------------------------------------------------+--------------------------------------------------+
|personid                            |tenant|birthdate                 |zipcode |gender            |birthsex          |maritalstatus                   |race                                                   |ethnicity                                         |
+------------------------------------+------+--------------------------+--------+------------------+------------------+--------------------------------+-------------------------------------------------------+--------------------------------------------------+
|00000007-4c76-45de-8b82-8052b3564944|13    |[[1981-06-19], 1981-06-19]|[[3], 3]|[[Female], Female]|[[Female], Female]|[[Never Married], Never Married]|[[White], White]                                       |[[Not Hispan

In [14]:
from datetime import datetime

def birthdate_transformation(input_string):
    b = "1963-06-28"  # Assuming the input_string is in the format "YYYY-MM-DD"
    formatted_date = datetime.strptime(b, "%Y-%m-%d").strftime("%B %d, %Y")
    return formatted_date

input_string = "[[1963-06-28], 1963-06-28]"
result = birthdate_transformation(input_string)
print(result)


June 28, 1963


In [144]:
# %%time
# df = pd.read_parquet('demographics_s.parquet', engine="pyarrow")

In [17]:
# %%time
# # !pip install fastparquet
# df = pd.read_parquet('clinical_event_s.parquet', engine="fastparquet")

▸,:,


In [143]:
# df.info()

In [142]:
# sparkdf = spark.createDataFrame(df)

In [158]:
demographics_table = spark.read.parquet('file:/home/z_miao/work/Oklahoma State/Zheng_Han/demographics_s.parquet')
demographics_table.createOrReplaceTempView("demographics_table")
allergy_table = spark.read.parquet('file:/home/z_miao/work/Oklahoma State/Zheng_Han/allergy_s.parquet')
allergy_table.createOrReplaceTempView("allergy_table")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [213]:
condition_table = spark.read.parquet('file:/home/z_miao/work/Oklahoma State/Zheng_Han/condition_s.parquet')
condition_table.createOrReplaceTempView("condition_table")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [214]:
condition_table.printSchema()

▸,:,


root
 |-- conditionid: string (nullable = true)
 |-- personid: string (nullable = true)
 |-- encounterid: string (nullable = true)
 |-- conditioncode: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodings: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- codingSystemId: string (nullable = true)
 |    |    |    |-- primaryDisplay: string (nullable = true)
 |-- effectivedate: string (nullable = true)
 |-- asserteddate: string (nullable = true)
 |-- type: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodi

In [219]:
stroke = spark.sql("""
SELECT DISTINCT personid, encounterid, effectivedate
FROM condition_table
WHERE conditioncode['standard']['id'] = 'I63.9'
""")

▸,:,


In [220]:
stroke.show(5)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+--------------------+--------------------+
|            personid|         encounterid|       effectivedate|
+--------------------+--------------------+--------------------+
|87f2f0c6-8252-444...|45b8a837-2705-463...|2020-02-25T19:00:...|
|de2091a2-1868-4cf...|33d2e010-3f35-466...|2019-08-18T19:00:...|
|a5b72f5f-2d3a-42d...|8c963c1c-697a-453...|2015-12-14T15:49:...|
|08d7eda8-681c-4d7...|7a64da68-38cf-4c4...|2020-07-03T17:00:...|
|9cb9894c-ed2a-44e...|97dcd08f-5b39-446...|2019-07-18T13:03:...|
+--------------------+--------------------+--------------------+
only showing top 5 rows



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [221]:
stroke.createOrReplaceTempView('stroke_table')

▸,:,


In [224]:
stroke_demo = spark.sql("""
select l.personid, l.encounterid, r.birthdate, r.gender['standard']['primaryDisplay'] as gender, r.races['standard']['primaryDisplay'][0] as race
from stroke_table as l
inner join
demographics_table as r
on
l.personid = r.personid
""")

▸,:,


In [225]:
stroke_demo.show(5)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------------------+--------------------+-------------------+------+--------------------+
|            personid|         encounterid|          birthdate|gender|                race|
+--------------------+--------------------+-------------------+------+--------------------+
|9d035af7-c339-49f...|f28c7367-7585-414...|1975-05-19T00:00:00|  Male|Black or African ...|
|c30dd3eb-7878-44e...|92a85cf8-bbe5-441...|1986-05-14T00:00:00|  Male|          Other Race|
|fbbadb41-0c46-44f...|d36e997d-ee93-4c9...|1945-04-12T00:00:00|  Male|Unknown racial group|
|b06046bc-0f78-4b0...|5497b92d-0c4c-44f...|1932-07-08T00:00:00|Female|Unknown racial group|
+--------------------+--------------------+-------------------+------+--------------------+



<IPython.core.display.Javascript object>

In [188]:
demographics_table.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- birthdate: string (nullable = true)
 |-- dateofdeath: string (nullable = true)
 |-- gender: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodings: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- codingSystemId: string (nullable = true)
 |    |    |    |-- primaryDisplay: string (nullable = true)
 |-- birthsex: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodings: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |

In [189]:
demographics_table.show(5)

▸,:,


<IPython.core.display.Javascript object>

+--------------------+-------------------+-----------+--------------------+--------------------+-------------+--------------------+--------------------+--------+-------+--------+------+------+
|            personid|          birthdate|dateofdeath|              gender|            birthsex|maritalstatus|               races|         ethnicities|deceased| source|zip_code|active|tenant|
+--------------------+-------------------+-----------+--------------------+--------------------+-------------+--------------------+--------------------+--------+-------+--------+------+------+
|5bcc348c-e279-49e...|1957-07-25T00:00:00|           |[[F, 2.16.840.1.1...|                 [,]|          [,]|[[[2106-3, 2.16.8...|[[[2186-5, 2.16.8...|   false|BILLING|        |  true|   115|
|ecb015fd-4fea-452...|2013-06-26T00:00:00|           |[[M, 2.16.840.1.1...|[[M, 2.16.840.1.1...|          [,]|[[[2106-3, 2.16.8...|[[, []], [[2186-5...|   false|    EMR|       5|  true|   115|
|a58b0729-896f-462...|1998-04-28T00

<IPython.core.display.Javascript object>

In [176]:
allergy_1 = spark.sql("""
SELECT DISTINCT personid, encounterid, criticality['standard']['primaryDisplay'] AS criticality
FROM allergy_table
WHERE substring(onsetdate, 1, 4) >= "2017"
"""
)
allergy_1.createOrReplaceTempView("allergy_1")

▸,:,


In [163]:
allergy_1.rdd.getNumPartitions()

▸,:,


200

In [ ]:
allergy_demographics = spark

In [209]:
allergy_demographics = spark.sql("""
SELECT l.personid, l.encounterid, r.gender['standard']['primaryDisplay'], l.criticality
FROM allergy_1 AS l
INNER JOIN
demographics_table as r
on l.personid = r.personid
WHERE l.criticality = 'Low'
""")

▸,:,


In [198]:
allergy_demographics = allergy_1.join(demographics,'personid','inner').select()

▸,:,


In [212]:
%%time
allergy_demographics.count()

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

CPU times: user 150 ms, sys: 19.8 ms, total: 170 ms
Wall time: 1min 4s


1

<IPython.core.display.Javascript object>

In [126]:
# df.printSchema()

root
 |-- personid: string (nullable = true)
 |-- birthdate: string (nullable = true)
 |-- dateofdeath: string (nullable = true)
 |-- gender: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodings: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- codingSystemId: string (nullable = true)
 |    |    |    |-- primaryDisplay: string (nullable = true)
 |-- birthsex: struct (nullable = true)
 |    |-- standard: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- codingSystemId: string (nullable = true)
 |    |    |-- primaryDisplay: string (nullable = true)
 |    |-- standardCodings: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |

In [141]:
# !pwd

In [127]:
# %%time
# import pyarrow as pa
# import pyarrow.parquet as pq
# df = pq.read_table('demographics_s.parquet')

In [129]:
# df

In [130]:
# !pip install duckdb

In [131]:
# import duckdb
# con = duckdb.connect()
# # demographics_table = pq.read_table('demographics_s.parquet')
# demographics_table = pd.read_parquet('demographics_s.parquet')

In [132]:
# demographics_table

In [133]:
# # allergy_table = pq.read_table('allergy_s.parquet')
# allergy_table = pd.read_parquet('allergy_s.parquet')

In [134]:
# allergy_table

In [135]:
# allergy_personid = con.execute("""
# SELECT DISTINCT personid, criticality.standard.primaryDisplay AS criticality
# FROM allergy_table
# WHERE substring(onsetdate, 1, 4) >= '2017'
# """).df()

In [136]:
# from pandasql import sqldf
# allergy_personid = sqldf("""
# SELECT DISTINCT personid, criticality['standard']['primaryDisplay'] AS criticality
# FROM allergy_table
# WHERE substring(onsetdate, 1, 4) >= '2017'
# """)

In [137]:
# allergy_personid

In [138]:
# allergy_demographics = con.execute("""
# SELECT allergy_personid.encounterid, gender['standard']['primaryDisplay'] AS gender
# FROM demographics_table LEFT JOIN allergy_personid on allergy_personid.encounterid
# """).df()

In [139]:
# personid.to_pandas()

In [34]:
# import dask.dataframe as dd
# df = dd.read_parquet('demographics_s.parquet')

In [41]:
# pandas_df = df.compute()
# pyspark_df = spark.createDataFrame(pandas_df) 

In [42]:
# %%time
# df = df.to_pandas('demographics_s.parquet')

In [140]:
# %%time
# sparkdf = spark.createDataFrame(df)

In [151]:
pixiedust.enableJobMonitor()

Successfully enabled Spark Job Progress Monitor


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>